In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss

%store -r df
adf_result = adfuller(df['CLOSE'].dropna(), regression='c', result_object=True)
kpss_result = kpss(df['CLOSE'].dropna(), regression='c', result_object=True )
print('--------- ADF - test  -------------- ')
print(adf_result)
print('--------- KPSS - test  -------------- ')
print(kpss_result)

--------- ADF - test  -------------- 
ADFullerResult
ADF Statistic: -2.29422
P-value: 0.17384
Used Lag: 18
Nobs: 738
Critical Values: {'1%': np.float64(-3.4392418578608983), '5%': np.float64(-2.86546426916434), '10%': np.float64(-2.568859710342903)}

--------- KPSS - test  -------------- 
KPSSResult
KPSS Statistic: 0.65018
P-value: 0.01000
Lags: 17
Critical Values: {'10%': 0.119, '5%': 0.146, '2.5%': 0.176, '1%': 0.216}



C:\Users\user\AppData\Local\Temp\ipykernel_6972\1122923531.py:7: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_result = kpss(df['CLOSE'].dropna(), regression='ct', result_object=True )


In [43]:
def find_stationary_windows(
    df,
    column_name,
    min_pct=0.20,
    max_pct=0.70,
    step_pct=0.05,
    alpha_adf=0.05,
    alpha_kpss=0.05,):
    """Проверяет временной ряд скользящими окнами разной длины с помощью ADF и KPSS.

    Сохраняет только те окна, где подтверждена стационарность.
    """
    series = df[column_name].dropna()
    total_len = len(series)
    successful_windows = []

    # 1. Переводим проценты в абсолютные значения размеров окон
    min_window = int(total_len * min_pct)
    max_window = int(total_len * max_pct)
    step = max(1, int(total_len * step_pct))  # Шаг изменения размера окна в барах

    window_sizes = range(min_window, max_window + 1, step)

    print(f"Анализ ряда длиной {total_len}. Тестируются окна от {min_window} до {max_window} баров.")

    # 2. Проход по всем размерам окон
    for window_size in window_sizes:
        # Скользящее окно по всему ряду
        for start_idx in range(0, total_len - window_size + 1):
            end_idx = start_idx + window_size
            window_data = series.iloc[start_idx:end_idx]

            # Исключаем окна с константными значениями (тесты упадут с ошибкой)
            if window_data.nunique() <= 1:
                continue

            try:
                # ADF-тест (H0: Ряд нестационарен)
                # Нужен p-value < alpha_adf, чтобы ОТВЕРГНУТЬ нестационарность
                adf_res = adfuller(window_data, regression="c", autolag="AIC")
                adf_p = adf_res[1]

                # KPSS-тест (H0: Ряд стационарен)
                # Нужен p-value >= alpha_kpss, чтобы НЕ ОТВЕРГАТЬ стационарность
                kpss_res = kpss(window_data, regression="c", nlags="auto")
                kpss_p = kpss_res[1]

                # 3. Проверка критерия двойной стационарности
                if adf_p < alpha_adf and kpss_p >= alpha_kpss:
                    successful_windows.append(
                        {
                            "window_size": window_size,
                            "window_pct": round(window_size / total_len, 2),
                            "start_index": series.index[start_idx],
                            "end_index": series.index[end_idx - 1],
                            "adf_p_value": adf_p,
                            "kpss_p_value": kpss_p,
                            "adf_stat": adf_res[0],
                            "kpss_stat": kpss_res[0],
                        }
                    )
            except Exception:
                # Игнорируем ошибки сходимости тестов на специфических выборках
                continue

    # Возвращаем результат в виде структурированного DataFrame
    return pd.DataFrame(successful_windows)

results_df = find_stationary_windows(df=df, column_name="CLOSE", min_pct=0.30, max_pct=0.70, step_pct=0.05)
results_df

Анализ ряда длиной 757. Тестируются окна от 227 до 529 баров.


C:\Users\user\AppData\Local\Temp\ipykernel_6972\2313538808.py:40: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_res = adfuller(window_data, regression="c", autolag="AIC")
C:\Users\user\AppData\Local\Temp\ipykernel_6972\2313538808.py:45: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_res = kpss(window_data, regression="c", nlags="auto")
C:\Users\user\AppData\Local\Temp\ipykernel_6972\2313538808.py:45: FutureWarning: kpss currently returns a plain tuple whose length and layout depends on the store argument (and which silently drops `lags` when store=Tru

,window_size,window_pct,start_index,end_index,adf_p_value,kpss_p_value,adf_stat,kpss_stat
0,264,0.35,295,558,0.017509,0.050711,-3.245278,0.461350
1,264,0.35,296,559,0.031801,0.057913,-3.034439,0.444642
2,264,0.35,297,560,0.024544,0.065076,-3.128070,0.428023
3,264,0.35,337,600,0.049007,0.100000,-2.869533,0.152950
4,301,0.40,290,590,0.005410,0.062191,-3.618909,0.434717
5,301,0.40,294,594,0.022575,0.090999,-3.157579,0.367883
6,301,0.40,297,597,0.045794,0.100000,-2.896178,0.331339
7,523,0.69,84,606,0.042689,0.069356,-2.923444,0.418093
8,523,0.69,85,607,0.036002,0.068162,-2.988294,0.420865
9,523,0.69,87,609,0.037379,0.064769,-2.974157,0.428735


In [48]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss


def find_stationary_windows(
    df,
    column_name,
    min_pct=0.20,
    max_pct=0.70,
    step_pct=0.05,
    alpha_adf=0.05,
    alpha_kpss=0.05,
):
    """Проверяет временной ряд скользящими окнами разной длины с помощью ADF и KPSS.

    Сохраняет только те окна, где подтверждена стационарность, включая даты начала и конца.
    """
    series = df[column_name].dropna()
    total_len = len(series)
    successful_windows = []

    # 1. Переводим проценты в абсолютные значения размеров окон
    min_window = int(total_len * min_pct)
    max_window = int(total_len * max_pct)
    step = max(1, int(total_len * step_pct))  # Шаг изменения размера окна в барах
    window_sizes = range(min_window, max_window + 1, step)

    print(
        f"Анализ ряда длиной {total_len}. Тестируются окна от {min_window} до {max_window} баров."
    )

    # 2. Проход по всем размерам окон
    for window_size in window_sizes:
        # Скользящее окно по всему ряду
        for start_idx in range(0, total_len - window_size + 1):
            end_idx = start_idx + window_size
            window_data = series.iloc[start_idx:end_idx]

            # Исключаем окна с константными значениями (тесты упадут с ошибкой)
            if window_data.nunique() <= 1:
                continue

            try:
                # ADF-тест (H0: Ряд нестационарен)
                adf_res = adfuller(window_data, regression="c", autolag="AIC")
                adf_p = adf_res[1]

                # KPSS-тест (H0: Ряд стационарен)
                kpss_res = kpss(window_data, regression="c", nlags="auto")
                kpss_p = kpss_res[1]

                # 3. Проверка критерия двойной стационарности
                if adf_p < alpha_adf and kpss_p >= alpha_kpss:
                    # Получаем метки индекса для корректного поиска в df
                    start_label = series.index[start_idx]
                    end_label = series.index[end_idx - 1]

                    successful_windows.append(
                        {
                            "window_size": window_size,
                            "window_pct": round(window_size / total_len, 2),
                            "start_index": start_label,
                            "end_index": end_label,
                            # Добавляем поиск по DATETIME с помощью .loc
                            "start_datetime": df.loc[start_label, "DATETIME"],
                            "end_datetime": df.loc[end_label, "DATETIME"],
                            "adf_p_value": adf_p,
                            "kpss_p_value": kpss_p,
                            "adf_stat": adf_res[0],
                            "kpss_stat": kpss_res[0],
                        }
                    )
            except Exception:
                # Игнорируем ошибки сходимости тестов на специфических выборках
                continue

    # Возвращаем результат в виде структурированного DataFrame
    return pd.DataFrame(successful_windows)


# Пример вызова функции
results_df = find_stationary_windows(
    df=df, column_name="CLOSE", min_pct=0.30, max_pct=0.70, step_pct=0.05
)
results_df

Анализ ряда длиной 757. Тестируются окна от 227 до 529 баров.


C:\Users\user\AppData\Local\Temp\ipykernel_6972\1496183313.py:45: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_res = adfuller(window_data, regression="c", autolag="AIC")
C:\Users\user\AppData\Local\Temp\ipykernel_6972\1496183313.py:49: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_res = kpss(window_data, regression="c", nlags="auto")
C:\Users\user\AppData\Local\Temp\ipykernel_6972\1496183313.py:49: FutureWarning: kpss currently returns a plain tuple whose length and layout depends on the store argument (and which silently drops `lags` when store=Tru

,window_size,window_pct,start_index,end_index,start_datetime,end_datetime,adf_p_value,kpss_p_value,adf_stat,kpss_stat
0,264,0.35,295,558,2026-09-23 18:00:00,2026-10-12 21:00:00,0.017509,0.050711,-3.245278,0.461350
1,264,0.35,296,559,2026-09-23 19:00:00,2026-10-12 22:00:00,0.031801,0.057913,-3.034439,0.444642
2,264,0.35,297,560,2026-09-23 20:00:00,2026-10-12 23:00:00,0.024544,0.065076,-3.128070,0.428023
3,264,0.35,337,600,2026-09-26 12:00:00,2026-10-15 11:00:00,0.049007,0.100000,-2.869533,0.152950
4,301,0.40,290,590,2026-09-23 13:00:00,2026-10-14 18:00:00,0.005410,0.062191,-3.618909,0.434717
5,301,0.40,294,594,2026-09-23 17:00:00,2026-10-14 22:00:00,0.022575,0.090999,-3.157579,0.367883
6,301,0.40,297,597,2026-09-23 20:00:00,2026-10-15 08:00:00,0.045794,0.100000,-2.896178,0.331339
7,523,0.69,84,606,2026-09-07 21:00:00,2026-10-15 17:00:00,0.042689,0.069356,-2.923444,0.418093
8,523,0.69,85,607,2026-09-08 07:00:00,2026-10-15 18:00:00,0.036002,0.068162,-2.988294,0.420865
9,523,0.69,87,609,2026-09-08 09:00:00,2026-10-15 20:00:00,0.037379,0.064769,-2.974157,0.428735


In [19]:
def adf_rolling(x):
    return adfuller(x, regression='c', autolag="AIC", result_object=True).pvalue
def kpss_rolling(x):
    return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
# window = 300
# df['ADF roll'] = df['<CLOSE>'].rolling(window).apply(adf_rolling, raw=True)
# df['KPSS roll'] = df['<CLOSE>'].rolling(window).apply(kpss_rolling, raw=True)
# df.iloc[:61]

In [20]:
def stationarity_results_to_df (df, min_pct=0.20, max_pct=0.70, step_pct=0.05):
    total_len = len(df['CLOSE'])
    min_window = int(min_pct * total_len)
    max_window = int(max_pct * total_len)
    step_window = max(1, int(step_pct * total_len))
    window_sizes = range(min_window, max_window + 1, step_window)

    for window in window_sizes:
        df[f'ADF {window}'] = df['CLOSE'].rolling(window).apply(adf_rolling, raw=True)
        df[f'KPSS {window}'] = df['CLOSE'].rolling(window).apply(kpss_rolling, raw=True)

    return df

In [21]:
min_pct = float(input('enter min % in decimal format'))
max_pct = float(input('enter max bond for percentage in decimal format'))
step_pct = float(input('enter step % in decimal format'))

new_df = stationarity_results_to_df(df, min_pct, max_pct, step_pct)

C:\Users\user\AppData\Local\Temp\ipykernel_6972\3036405258.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
C:\Users\user\AppData\Local\Temp\ipykernel_6972\3036405258.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
C:\Users\user\AppData\Local\Temp\ipykernel_6972\3036405258.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
C:\Users\user\AppData\Local\Temp\ipykernel_6972\3036405258.py:4: InterpolationWarning

In [35]:
new_df

,TICKER,DATETIME,CLOSE,h-cleaned,ADF 151,KPSS 151,ADF 188,KPSS 188,ADF 225,KPSS 225,...,ADF 373,KPSS 373,ADF 410,KPSS 410,ADF 447,KPSS 447,ADF 484,KPSS 484,ADF 521,KPSS 521
0,SVZ6SVH7 [FUTSPREAD],2026-09-01 07:00:00,1.71,1.75,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SVZ6SVH7 [FUTSPREAD],2026-09-01 08:00:00,1.69,1.75,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SVZ6SVH7 [FUTSPREAD],2026-09-01 09:00:00,1.69,1.75,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SVZ6SVH7 [FUTSPREAD],2026-09-01 10:00:00,1.72,1.75,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SVZ6SVH7 [FUTSPREAD],2026-09-01 11:00:00,1.75,1.75,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
752,SVZ6SVH7 [FUTSPREAD],2026-10-27 13:00:00,1.56,1.56,0.418828,0.012368,0.092135,0.010000,0.297414,0.01,...,0.072409,0.01,0.069375,0.01,0.050719,0.01,0.022461,0.01,0.055176,0.01
753,SVZ6SVH7 [FUTSPREAD],2026-10-27 14:00:00,1.54,1.56,0.488624,0.017325,0.121818,0.010000,0.280122,0.01,...,0.055042,0.01,0.056517,0.01,0.037450,0.01,0.017419,0.01,0.045228,0.01
754,SVZ6SVH7 [FUTSPREAD],2026-10-27 15:00:00,1.54,1.56,0.516960,0.021155,0.177985,0.010000,0.145188,0.01,...,0.058365,0.01,0.042536,0.01,0.034043,0.01,0.014497,0.01,0.036962,0.01
755,SVZ6SVH7 [FUTSPREAD],2026-10-27 16:00:00,1.52,1.56,0.545848,0.026391,0.157302,0.015946,0.180553,0.01,...,0.058422,0.01,0.034185,0.01,0.035036,0.01,0.012687,0.01,0.042765,0.01


In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

price = go.Scatter(x=df['DATETIME'], y=df['h-cleaned'], name=df['TICKER'][1])
ADF_pvalue_window = go.Scatter(x=df['DATETIME'], y=df['ADF 225'], name='ADF rolling')
KPSS_pvalue_window = go.Scatter(x=df['DATETIME'], y=df['KPSS 225'], name='KPSS rolling')
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(price, secondary_y=False)
fig.add_trace(ADF_pvalue_window, secondary_y=True)
fig.add_trace(KPSS_pvalue_window, secondary_y=True)
fig


In [49]:
df_filtered = new_df[new_df['DATETIME'].between("2026-09-07 21:00:00", "2026-10-15 17:00:00	")]
# df_filtered = new_df.iloc[91:613]

# Выбираем целевые колонки для анализа (с 5-й и до конца)
target_cols = df_filtered.iloc[:, 4:]

# Находим все уникальные суффиксы
suffixes = set(col.split()[-1] for col in target_cols.columns if " " in col)

# Список для сбора колонок, которые нужно УДАЛИТЬ
cols_to_drop = []

# 2. Проверяем условия попарно
for suf in suffixes:
    adf_col = f"ADF {suf}"
    kpss_col = f"KPSS {suf}"

    if adf_col in target_cols.columns and kpss_col in target_cols.columns:
        # Проверяем условие (ADF < 0.05 и KPSS > 0.05 одновременно во всем периоде)
        condition = (target_cols[adf_col] < 0.05) & (target_cols[kpss_col] > 0.05)

        # Если условие НЕ выполнилось хотя бы в одной точке периода,
        # добавляем обе колонки в список на удаление
        if not condition.all():
            cols_to_drop.extend([adf_col, kpss_col])
    else:
        # Если какой-то колонки из пары изначально нет,
        # можно тоже добавить существующую в список на удаление
        if adf_col in target_cols.columns:
            cols_to_drop.append(adf_col)
        if kpss_col in target_cols.columns:
            cols_to_drop.append(kpss_col)

# 3. Удаляем неподходящие колонки из ИСХОДНОГО датафрейма df
# Исходный df сохраняет все свои строки (весь период времени) и первые 4 колонки
df_clean = new_df.drop(columns=cols_to_drop)
df_clean

,TICKER,DATETIME,CLOSE,h-cleaned
0,SVZ6SVH7 [FUTSPREAD],2026-09-01 07:00:00,1.71,1.75
1,SVZ6SVH7 [FUTSPREAD],2026-09-01 08:00:00,1.69,1.75
2,SVZ6SVH7 [FUTSPREAD],2026-09-01 09:00:00,1.69,1.75
3,SVZ6SVH7 [FUTSPREAD],2026-09-01 10:00:00,1.72,1.75
4,SVZ6SVH7 [FUTSPREAD],2026-09-01 11:00:00,1.75,1.75
...,...,...,...,...
752,SVZ6SVH7 [FUTSPREAD],2026-10-27 13:00:00,1.56,1.56
753,SVZ6SVH7 [FUTSPREAD],2026-10-27 14:00:00,1.54,1.56
754,SVZ6SVH7 [FUTSPREAD],2026-10-27 15:00:00,1.54,1.56
755,SVZ6SVH7 [FUTSPREAD],2026-10-27 16:00:00,1.52,1.56


In [38]:
df_clean.columns

Index(['TICKER', 'DATETIME', 'CLOSE', 'h-cleaned', 'ADF 151', 'KPSS 151',
       'ADF 188', 'KPSS 188', 'ADF 225', 'KPSS 225', 'ADF 262', 'KPSS 262',
       'ADF 299', 'KPSS 299', 'ADF 336', 'KPSS 336', 'ADF 373', 'KPSS 373',
       'ADF 410', 'KPSS 410', 'ADF 447', 'KPSS 447', 'ADF 484', 'KPSS 484',
       'ADF 521', 'KPSS 521'],
      dtype='str')

In [55]:
res = adfuller(df['CLOSE'][86:608], regression='c', autolag="AIC", result_object=True)
res

ADFullerResult
ADF Statistic: -2.78231
P-value: 0.06085
Used Lag: 18
Nobs: 503
Critical Values: {'1%': np.float64(-3.4434175660489905), '5%': np.float64(-2.8673031724657454), '10%': np.float64(-2.5698395516760275)}

In [57]:
import os
import pandas as pd
from joblib import Parallel, delayed
from statsmodels.tsa.stattools import adfuller, kpss


def _check_single_window(window_data, start_label, end_label, w_size, total_len):
    # если данные почти не меняются, тесты выдадут ошибку
    if window_data.nunique() <= 1:
        return None

    try:
        adf_res = adfuller(window_data, autolag="AIC")
        if adf_res[1] >= 0.05:
            return None

        #(запускается только если ADF пройден)
        kpss_res = kpss(window_data, regression="c", nlags="auto")
        if kpss_res[1] < 0.05:
            return None
        # Если оба теста пройдены
        return {"window_size": w_size,
                "window_pct": round(w_size / total_len, 2),
                "start_index": start_label,
                "end_index": end_label,
                "adf_p_value": adf_res[1],
                "kpss_p_value": kpss_res[1]}
    except Exception:
        return None


def find_stationary_windows_fast(df, column_name, min_pct=0.20, max_pct=0.70, step_pct=0.05, n_jobs=-1):
    series = df[column_name].dropna()
    total_len = len(series)

    # Генерируем параметры всех окон (размеры и индексы)
    min_window = int(total_len * min_pct)
    max_window = int(total_len * max_pct)
    step = max(1, int(total_len * step_pct))

    window_sizes = range(min_window, max_window + 1, step)

    # Создаем список задач (генератор), чтобы не забивать память
    tasks = []
    for w_size in window_sizes:
        for start_idx in range(0, total_len - w_size + 1):
            end_idx = start_idx + w_size
            tasks.append((
                    series.iloc[start_idx:end_idx],
                    series.index[start_idx],
                    series.index[end_idx - 1],
                    w_size))

    print(f"Запущено параллельное вычисление для {len(tasks)} окон на {os.cpu_count() if n_jobs == -1 else n_jobs} ядрах...")

    # параллельно на всех ядрах процессора
    # delayed() оборачивает функцию, а Parallel распределяет таски по процессам
    results = (Parallel(n_jobs=n_jobs, backend="multiprocessing")
               (delayed(_check_single_window)(w_data, s_label, e_label, w_size, total_len)
                for w_data, s_label, e_label, w_size in tasks))
    # фильтруем пустые результаты и собираем дф
    successful_windows = [r for r in results if r is not None]

    return pd.DataFrame(successful_windows)

fast_results = find_stationary_windows_fast(
        df,
        "CLOSE",
        min_pct=0.20,
        max_pct=0.70,
        step_pct=0.05,
        n_jobs=-1)

fast_results

Запущено параллельное вычисление для 4642 окон на 4 ядрах...


KeyboardInterrupt: 